In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.lines as mlines
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
import matplotlib.colors as mcolors

# Archivos para la versión absoluta
file_abs_age = "D:/MulticentersEEG/Features_2_normativeModel/gamma_40/24_BEST_EPOCHS/AIF_Babiloni/results_harmonize/neuroharmonize/osc_pw_ab_canonic_roiPO_SITE_age_neuroharmonize.xlsx"
file_abs_age_sex = "D:/MulticentersEEG/Features_2_normativeModel/gamma_40/24_BEST_EPOCHS/AIF_Babiloni/results_harmonize/neuroharmonize/osc_pw_ab_canonic_roiPO_SITE_age_sex_neuroharmonize.xlsx"

# Archivos para la versión relativa
file_rel_age = "D:/MulticentersEEG/Features_2_normativeModel/gamma_40/24_BEST_EPOCHS/AIF_Babiloni/results_harmonize/neuroharmonize/osc_pw_rel_canonic_roiPO_SITE_age_neuroharmonize.xlsx"
file_rel_age_sex = "D:/MulticentersEEG/Features_2_normativeModel/gamma_40/24_BEST_EPOCHS/AIF_Babiloni/results_harmonize/neuroharmonize/osc_pw_rel_canonic_roiPO_SITE_age_sex_neuroharmonize.xlsx"

# Cargar dataframes
df_abs_age_unharm = pd.read_excel(file_abs_age, sheet_name="unharmonizeSITE_age")
df_abs_age_harm = pd.read_excel(file_abs_age, sheet_name="harmonizeSITE_age")
df_abs_age_sex_unharm = pd.read_excel(file_abs_age_sex, sheet_name="unharmonizeSITE_age_sex")
df_abs_age_sex_harm = pd.read_excel(file_abs_age_sex, sheet_name="harmonizeSITE_age_sex")

df_rel_age_unharm = pd.read_excel(file_rel_age, sheet_name="unharmonizeSITE_age")
df_rel_age_harm = pd.read_excel(file_rel_age, sheet_name="harmonizeSITE_age")
df_rel_age_sex_unharm = pd.read_excel(file_rel_age_sex, sheet_name="unharmonizeSITE_age_sex")
df_rel_age_sex_harm = pd.read_excel(file_rel_age_sex, sheet_name="harmonizeSITE_age_sex")

# Crear etiquetas de sitio y una paleta única
site_names = sorted(df_rel_age_unharm["SITE"].unique())
site_labels = {name: str(i) for i, name in enumerate(site_names)}
inv_site_labels = {v: k for k, v in site_labels.items()}  # Para revertir si lo necesitas

# Paleta única para los sitios, con claves numéricas (str)
palette_vals = sns.color_palette("tab20", len(site_labels))
palette_site = {site_labels[name]: color for name, color in zip(site_names, palette_vals)}

group_palette = {
    "HC": "blue",   # azul (puedes cambiarlo si quieres)
    "MCI": "red"   # naranja
}
# Mapeo de columnas PCA
pca_cols = {
    "unharmonized": ("pca-one", "pca-two"),
    "harmonized": ("harmonized-pca-one", "harmonized-pca-two")
}

def map_site_labels(df):
    unique_sites = sorted(df["SITE"].dropna().unique())
    local_labels = {name: str(i) for i, name in enumerate(unique_sites)}
    df = df.copy()
    df["SITE_num"] = df["SITE"].map(local_labels)
    return df, local_labels

def compute_centroid_distance_matrix(df, x_col, y_col, label_col):
    labels = sorted(df[label_col].unique())
    centroids = df.groupby(label_col)[[x_col, y_col]].mean()
    dist_matrix = pd.DataFrame(index=labels, columns=labels, dtype=float)
    for i in labels:
        for j in labels:
            dist_matrix.loc[i, j] = np.linalg.norm(centroids.loc[i] - centroids.loc[j])
    return dist_matrix


def compute_difference_sign_matrix(harm_matrix, unharm_matrix, epsilon=0.01):
    diff = harm_matrix.astype(float) - unharm_matrix.astype(float)
    sign_matrix = np.zeros_like(diff, dtype=int)
    sign_matrix[diff < -epsilon] = -1
    sign_matrix[diff > epsilon] = 1
    return diff, sign_matrix

def plot_change_matrix(ax, diff_sign, diff_vals, title="Change Matrix"):
    mask = np.triu(np.ones_like(diff_sign, dtype=bool))
    cmap = mcolors.ListedColormap(['thistle', 'white', 'indigo'])
    bounds = [-1.5, -0.5, 0.5, 1.5]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)
    annot = diff_vals.copy()
    annot[(diff_sign != 1)] = ""
    annot[diff_vals.isna()] = ""

    sns.heatmap(diff_sign, mask=mask, cmap=cmap, norm=norm, annot=annot,
                fmt="", annot_kws={"size": 5}, square=True, cbar=False, ax=ax,linewidths=0, linecolor=None)
    ax.set_title(title)
    ax.set_xlabel("SITE")
    ax.set_ylabel("SITE")
    legend_elements = [
        Patch(facecolor='indigo', label='Increased'),
        Patch(facecolor='thistle', label='Decreased'),
        Patch(facecolor='white', label='No change')
    ]
    ax.legend(handles=legend_elements, title="Change", loc="upper left", bbox_to_anchor=(1, 1))
    ax.grid(False)

import seaborn as sns
import matplotlib.lines as mlines

def plot_pca(ax, df, x_col, y_col, label_col, title, palette, return_legend=False):
    unique_labels = df[label_col].unique()
    
    # Puntos individuales
    sns.scatterplot(
        data=df, x=x_col, y=y_col, hue=label_col,
        ax=ax, palette=palette, s=10, alpha=0.2, legend=False
    )

    # Centroides con borde del mismo color
    centroids = df.groupby(label_col)[[x_col, y_col]].mean()
    for label in centroids.index:
        ax.scatter(
            centroids.loc[label, x_col], centroids.loc[label, y_col],
            s=60, color=palette[label], edgecolor=palette[label]
        )

    # Cruces punteadas en el centro
    ax.axvline(0, color='gray', linestyle='--', linewidth=1)
    ax.axhline(0, color='gray', linestyle='--', linewidth=1)
    if title == "PCA by GROUP":
        counts = df[label_col].value_counts()
        n_HC = counts.get("HC", 0)
        n_MCI = counts.get("MCI", 0)
        title = f"PCA by GROUP\nHC n={n_HC} | MCI n={n_MCI}"

    # Estética
    ax.set_title(title)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])

    if return_legend:
        # Ordenar por número si son dígitos
        if all(str(label).isdigit() for label in unique_labels):
            sorted_labels = sorted(unique_labels, key=lambda x: int(x))
            legend_title = "SITE"
        else:
            sorted_labels = sorted(unique_labels)
            legend_title = "GROUP"

        handles = [
            mlines.Line2D([], [], color=palette[label], marker='o', linestyle='None',
                          markersize=7, label=str(label))
            for label in sorted_labels
        ]
        return handles, legend_title


# Prepara la figura
fig, axes = plt.subplots(5, 4, figsize=(16, 12))  # 4x4 + 1x4

site_names_abs = set(df_abs_age_unharm["SITE"].unique()).union(df_abs_age_sex_unharm["SITE"].unique())
site_names_rel = set(df_rel_age_unharm["SITE"].unique()).union(df_rel_age_sex_unharm["SITE"].unique())
site_names = sorted(site_names_abs.union(site_names_rel))

site_labels = {name: str(i) for i, name in enumerate(site_names)}

# Definir dataset relativos y absolutos
datasets = {
    "abs": [
        (df_abs_age_unharm, "unharmonized", "age"),
        (df_abs_age_harm, "harmonized", "age"),
        (df_abs_age_sex_unharm, "unharmonized", "age+sex"),
        (df_abs_age_sex_harm, "harmonized", "age+sex")
    ],
    "rel": [
        (df_rel_age_unharm, "unharmonized", "age"),
        (df_rel_age_harm, "harmonized", "age"),
        (df_rel_age_sex_unharm, "unharmonized", "age+sex"),
        (df_rel_age_sex_harm, "harmonized", "age+sex")
    ]
}

# Plot PCA
for i in range(4):
    for j, key in enumerate(["abs", "rel"]):
        df, harmon, covar = datasets[key][i]
        x_pca, y_pca = pca_cols["harmonized"] if harmon == "harmonized" else pca_cols["unharmonized"]
        df, labels= map_site_labels(df) 

        if i == 0:
            site_handles, _ = plot_pca(axes[i, j * 2], df, x_pca, y_pca, "SITE_num", "PCA by SITE", palette_site, return_legend=True)
            group_handles, _ = plot_pca(axes[i, j * 2 + 1], df, x_pca, y_pca, "group", "PCA by GROUP", group_palette, return_legend=True)
        else:
            plot_pca(axes[i, j * 2], df, x_pca, y_pca, "SITE_num", "PCA by SITE", palette_site)
            plot_pca(axes[i, j * 2 + 1], df, x_pca, y_pca, "group", "PCA by GROUP", group_palette)





df1, labels1  = map_site_labels(df_abs_age_unharm)
df2, labels2 = map_site_labels(df_abs_age_harm)
x1, y1 = pca_cols["unharmonized"]
x2, y2 = pca_cols["harmonized"]
m1 = compute_centroid_distance_matrix(df1, x1, y1, "SITE_num",)
m2 = compute_centroid_distance_matrix(df2, x2, y2, "SITE_num",)

diff_vals, diff_sign = compute_difference_sign_matrix(m2, m1)
plot_change_matrix(axes[4, 0], diff_sign, diff_vals.round(2), title="Change Matrix (AGE)")


df3 , labels3= map_site_labels(df_abs_age_sex_unharm)
df4, labels4 = map_site_labels(df_abs_age_sex_harm)
m3 = compute_centroid_distance_matrix(df3, x1, y1, "SITE_num")
m4 = compute_centroid_distance_matrix(df4, x2, y2, "SITE_num")
diff_vals, diff_sign = compute_difference_sign_matrix(m4, m3)
plot_change_matrix(axes[4, 1], diff_sign, diff_vals.round(2), title="Change Matrix (AGE + SEX)")

# RELATIVE
df1, labels1 = map_site_labels(df_rel_age_unharm)
df2, labels2 = map_site_labels(df_rel_age_harm)
m1 = compute_centroid_distance_matrix(df1, x1, y1, "SITE_num",)
m2 = compute_centroid_distance_matrix(df2, x2, y2, "SITE_num",)

diff_vals, diff_sign = compute_difference_sign_matrix(m2, m1)
plot_change_matrix(axes[4, 2], diff_sign, diff_vals.round(2), title="Change Matrix (AGE)")

df3,labels3  = map_site_labels(df_rel_age_sex_unharm)
df4,labels4  = map_site_labels(df_rel_age_sex_harm)
m3 = compute_centroid_distance_matrix(df3, x1, y1, "SITE_num",)
m4 = compute_centroid_distance_matrix(df4, x2, y2, "SITE_num",)
diff_vals, diff_sign = compute_difference_sign_matrix(m4, m3)
plot_change_matrix(axes[4, 3], diff_sign, diff_vals.round(2), title="Change Matrix (AGE + SEX)")

plt.suptitle("Performance of Harmonization: Qualitative visualization, Neuroharmonize", fontsize=16, fontweight="bold")
# Encabezados para Absolute y Relative encima de las columnas
fig.text(0.22, 0.939, "Oscillatory Absolute power - ROI PO", ha='center', va='bottom', fontsize=12, fontweight='bold')
fig.text(0.73, 0.939, "Oscillatory Relative power - ROI PO", ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.97])
# Leyenda combinada (SITE + GROUP) en una sola fila

# Leyenda combinada (SITE + GROUP) en una sola fila

fig.legend(
    handles=site_handles + group_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.06),
    ncol=len(site_handles + group_handles),
    fontsize=10,
    frameon=False
)
site_names = sorted(df["SITE"].unique())
site_labels = {name: str(i) for i, name in enumerate(site_names)}
site_labels_inv = {v: k for k, v in site_labels.items()}

# Texto con la relación de etiquetas numéricas y nombres

legend_text = " | ".join([f"{v}: {k}" for k, v in labels1.items()])


fig.text(0.5, -0.03, "Site labels: " + legend_text,
         ha='center', va='center', fontsize=12)
row_labels = [
    "Unharmonized Age",
    "Harmonized Age",
    "Unharmonized Age+Sex",
    "Harmonized Age+Sex"
]

# Añadir etiquetas de fila a la izquierda de cada fila (fila 0 a 3)
for i, label in enumerate(row_labels):
    fig.text(-0.01, 0.84 - i * 0.19, label, va='center', ha='left', fontsize=12, fontweight='bold', rotation=90)

# Etiquetas tipo "Panel A", "Panel B", "Panel C"
fig.text(0.01, 0.94, "A", fontsize=18, fontweight='bold')  # izquierda arriba
fig.text(0.51, 0.94, "B", fontsize=18, fontweight='bold')  # derecha arriba
fig.text(0.01, 0.18, "C", fontsize=18, fontweight='bold')  # antes de última fila

plt.savefig("D:/MulticentersEEG/graphs_thesis_LMZS/performance_harmonization_neuroharmonize_OSC_PO.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.lines as mlines
import seaborn as sns
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist
import matplotlib.colors as mcolors

# Archivos para la versión absoluta
file_abs_age = "D:/MulticentersEEG/Features_2_normativeModel/gamma_40/24_BEST_EPOCHS/AIF_Babiloni/results_harmonize/recombat/osc_pw_ab_canonic_roiPO_SITE_age_group_recombat.xlsx"
file_abs_age_sex = "D:/MulticentersEEG/Features_2_normativeModel/gamma_40/24_BEST_EPOCHS/AIF_Babiloni/results_harmonize/recombat/osc_pw_ab_canonic_roiPO_SITE_age_sex_group_recombat.xlsx"

# Archivos para la versión relativa
file_rel_age = "D:/MulticentersEEG/Features_2_normativeModel/gamma_40/24_BEST_EPOCHS/AIF_Babiloni/results_harmonize/recombat/osc_pw_rel_canonic_roiPO_SITE_age_group_recombat.xlsx"
file_rel_age_sex = "D:/MulticentersEEG/Features_2_normativeModel/gamma_40/24_BEST_EPOCHS/AIF_Babiloni/results_harmonize/recombat/osc_pw_rel_canonic_roiPO_SITE_age_sex_group_recombat.xlsx"

# Cargar dataframes
df_abs_age_unharm = pd.read_excel(file_abs_age, sheet_name="unharmonizeSITE_age_group")
df_abs_age_harm = pd.read_excel(file_abs_age, sheet_name="harmonizeSITE_age_group")
df_abs_age_sex_unharm = pd.read_excel(file_abs_age_sex, sheet_name="unharmonizeSITE_age_sex_group")
df_abs_age_sex_harm = pd.read_excel(file_abs_age_sex, sheet_name="harmonizeSITE_age_sex_group")

df_rel_age_unharm = pd.read_excel(file_rel_age, sheet_name="unharmonizeSITE_age_group")
df_rel_age_harm = pd.read_excel(file_rel_age, sheet_name="harmonizeSITE_age_group")
df_rel_age_sex_unharm = pd.read_excel(file_rel_age_sex, sheet_name="unharmonizeSITE_age_sex_group")
df_rel_age_sex_harm = pd.read_excel(file_rel_age_sex, sheet_name="harmonizeSITE_age_sex_group")


# Crear etiquetas de sitio y una paleta única
site_names = sorted(df_rel_age_unharm["SITE"].unique())
site_labels = {name: str(i) for i, name in enumerate(site_names)}
inv_site_labels = {v: k for k, v in site_labels.items()}  # Para revertir si lo necesitas

# Paleta única para los sitios, con claves numéricas (str)
palette_vals = sns.color_palette("tab20", len(site_labels))
palette_site = {site_labels[name]: color for name, color in zip(site_names, palette_vals)}

group_palette = {
    "HC": "blue",   # azul (puedes cambiarlo si quieres)
    "MCI": "red"   # naranja
}
# Mapeo de columnas PCA
pca_cols = {
    "unharmonized": ("pca-one", "pca-two"),
    "harmonized": ("harmonized-pca-one", "harmonized-pca-two")
}



# Prepara la figura
fig, axes = plt.subplots(5, 4, figsize=(16, 12))  # 4x4 + 1x4

site_names_abs = set(df_abs_age_unharm["SITE"].unique()).union(df_abs_age_sex_unharm["SITE"].unique())
site_names_rel = set(df_rel_age_unharm["SITE"].unique()).union(df_rel_age_sex_unharm["SITE"].unique())
site_names = sorted(site_names_abs.union(site_names_rel))

site_labels = {name: str(i) for i, name in enumerate(site_names)}

# Definir dataset relativos y absolutos
datasets = {
    "abs": [
        (df_abs_age_unharm, "unharmonized", "age"),
        (df_abs_age_harm, "harmonized", "age"),
        (df_abs_age_sex_unharm, "unharmonized", "age+sex"),
        (df_abs_age_sex_harm, "harmonized", "age+sex")
    ],
    "rel": [
        (df_rel_age_unharm, "unharmonized", "age"),
        (df_rel_age_harm, "harmonized", "age"),
        (df_rel_age_sex_unharm, "unharmonized", "age+sex"),
        (df_rel_age_sex_harm, "harmonized", "age+sex")
    ]
}

# Plot PCA
for i in range(4):
    for j, key in enumerate(["abs", "rel"]):
        df, harmon, covar = datasets[key][i]
        x_pca, y_pca = pca_cols["harmonized"] if harmon == "harmonized" else pca_cols["unharmonized"]
        df, labels= map_site_labels(df) 

        if i == 0:
            site_handles, _ = plot_pca(axes[i, j * 2], df, x_pca, y_pca, "SITE_num", "PCA by SITE", palette_site, return_legend=True)
            group_handles, _ = plot_pca(axes[i, j * 2 + 1], df, x_pca, y_pca, "group", "PCA by GROUP", group_palette, return_legend=True)
        else:
            plot_pca(axes[i, j * 2], df, x_pca, y_pca, "SITE_num", "PCA by SITE", palette_site)
            plot_pca(axes[i, j * 2 + 1], df, x_pca, y_pca, "group", "PCA by GROUP", group_palette)


df1, labels1  = map_site_labels(df_abs_age_unharm)
df2, labels2 = map_site_labels(df_abs_age_harm)
x1, y1 = pca_cols["unharmonized"]
x2, y2 = pca_cols["harmonized"]
m1 = compute_centroid_distance_matrix(df1, x1, y1, "SITE_num",)
m2 = compute_centroid_distance_matrix(df2, x2, y2, "SITE_num",)

diff_vals, diff_sign = compute_difference_sign_matrix(m2, m1)
plot_change_matrix(axes[4, 0], diff_sign, diff_vals.round(2), title="Change Matrix (AGE + DIAGNOSIS)")


df3 , labels3= map_site_labels(df_abs_age_sex_unharm)
df4, labels4 = map_site_labels(df_abs_age_sex_harm)
m3 = compute_centroid_distance_matrix(df3, x1, y1, "SITE_num")
m4 = compute_centroid_distance_matrix(df4, x2, y2, "SITE_num")
diff_vals, diff_sign = compute_difference_sign_matrix(m4, m3)
plot_change_matrix(axes[4, 1], diff_sign, diff_vals.round(2), title="Change Matrix (AGE + SEX + DIAGNOSIS)")

# RELATIVE
df1, labels1 = map_site_labels(df_rel_age_unharm)
df2, labels2 = map_site_labels(df_rel_age_harm)
m1 = compute_centroid_distance_matrix(df1, x1, y1, "SITE_num",)
m2 = compute_centroid_distance_matrix(df2, x2, y2, "SITE_num",)

diff_vals, diff_sign = compute_difference_sign_matrix(m2, m1)
plot_change_matrix(axes[4, 2], diff_sign, diff_vals.round(2), title="Change Matrix (AGE + DIAGNOSIS)")

df3,labels3  = map_site_labels(df_rel_age_sex_unharm)
df4,labels4  = map_site_labels(df_rel_age_sex_harm)
m3 = compute_centroid_distance_matrix(df3, x1, y1, "SITE_num",)
m4 = compute_centroid_distance_matrix(df4, x2, y2, "SITE_num",)
diff_vals, diff_sign = compute_difference_sign_matrix(m4, m3)
plot_change_matrix(axes[4, 3], diff_sign, diff_vals.round(2), title="Change Matrix (AGE + SEX + DIAGNOSIS)")

plt.suptitle("Performance of Harmonization: Qualitative visualization, Recombat", fontsize=16, fontweight="bold")
# Encabezados para Absolute y Relative encima de las columnas
fig.text(0.22, 0.939, "Oscillatory Absolute power - ROI PO", ha='center', va='bottom', fontsize=12, fontweight='bold')
fig.text(0.73, 0.939, "Oscillatory Relative power - ROI PO", ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout(rect=[0, 0, 1, 0.97])
# Leyenda combinada (SITE + GROUP) en una sola fila

# Leyenda combinada (SITE + GROUP) en una sola fila

fig.legend(
    handles=site_handles + group_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.06),
    ncol=len(site_handles + group_handles),
    fontsize=10,
    frameon=False
)
site_names = sorted(df["SITE"].unique())
site_labels = {name: str(i) for i, name in enumerate(site_names)}
site_labels_inv = {v: k for k, v in site_labels.items()}

# Texto con la relación de etiquetas numéricas y nombres

legend_text = " | ".join([f"{v}: {k}" for k, v in labels1.items()])


fig.text(0.5, -0.03, "Site labels: " + legend_text,
         ha='center', va='center', fontsize=12)
row_labels = [
    "Unharmonized\nAge+Diagnosis",
    "Harmonized\nAge+Diagnosis",
    "Unharmonized\nAge+Sex+Diagnosis",
    "Harmonized\nAge+Sex+Diagnosis"
]

# Añadir etiquetas de fila a la izquierda de cada fila (fila 0 a 3)
for i, label in enumerate(row_labels):
    fig.text(-0.02, 0.84 - i * 0.19, label, va='center', ha='left', fontsize=12, fontweight='bold', rotation=90)

# Etiquetas tipo "Panel A", "Panel B", "Panel C"
fig.text(0.01, 0.94, "A", fontsize=18, fontweight='bold')  # izquierda arriba
fig.text(0.51, 0.94, "B", fontsize=18, fontweight='bold')  # derecha arriba
fig.text(0.01, 0.18, "C", fontsize=18, fontweight='bold')  # antes de última fila

plt.savefig("D:/MulticentersEEG/graphs_thesis_LMZS/performance_harmonization_recombat_OSC_PO.pdf", dpi=300, bbox_inches='tight')
#plt.savefig("/mnt/data/PCA_change_combined_absolute_relative.png", dpi=300)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import re

# Cargar el archivo
df = pd.read_excel(r"D:\MulticentersEEG\Features_2_normativeModel\gamma_40\24_BEST_EPOCHS\AIF_Babiloni\results_harmonize\neuroharmonize\statical_tests\tests.xlsx")
df_recombat = pd.read_excel(r"D:\MulticentersEEG\Features_2_normativeModel\gamma_40\24_BEST_EPOCHS\AIF_Babiloni\results_harmonize\recombat\statical_tests\tests.xlsx")

# Limpiar nombres de features
df['ROI'] = df['Feature'].str.extract(r'roi([A-Z]+)') 
df['Feature_clean'] = df['Feature'].apply(lambda x: re.sub(r'_?roi[A-Z]+_?', '', x))
df_age = df[df['covars'] == 'SITE_age']
df_sex = df[df['covars'] == 'SITE_age_sex']


df_recombat['ROI'] = df_recombat['Feature'].str.extract(r'roi([A-Z]+)') 
df_recombat['Feature_clean'] = df_recombat['Feature'].apply(lambda x: re.sub(r'_?roi[A-Z]+_?', '', x))
df_recombat_age = df_recombat[df_recombat['covars'] == 'SITE_age_group']
df_recombat_sex = df_recombat[df_recombat['covars'] == 'SITE_age_sex_group']


In [ ]:

# Definir condiciones
conditions = {
    "A) Unharmonized": df_age['levene_pval_zscore'],
    "B) Harmonized Age (Neuroharmonize)": df_age['levene_pval_Harmonized'],
    #"Unharmonized Age+Sex (Neuroharmonize)": df['levene_pval_zscore'],
    "C) Harmonized Age+Sex (Neuroharmonize)": df_sex['levene_pval_Harmonized'],
    #"Unharmonized Age + Diagnosis (ReComBat)": df_recombat_age['levene_pval_zscore'],
    "D) Harmonized Age + Diagnosis (ReComBat)": df_recombat_age['levene_pval_Harmonized'],
    #"Unharmonized Age + Sex + Diagnosis (ReComBat)": df_recombat['levene_pval_zscore'],
    "E) Harmonized Age + Sex + Diagnosis (ReComBat)": df_recombat_sex['levene_pval_Harmonized'] 
}

# Crear matrices binarias
binary_matrices = {}
for label, series in conditions.items():
    binarized = (series < 0.05).astype(int)
    binary_df = pd.DataFrame({
        'ROI': df['ROI'],
        'Feature': df['Feature_clean'],
        'Significant': binarized
    })
    binary_matrix = binary_df.pivot_table(index='ROI', columns='Feature', values='Significant', aggfunc='max').fillna(0)
    binary_matrices[label] = binary_matrix

# Definir orden y límites
features = [
    'IAFp', 'exponent',
    'pw_ab_theta', 'pw_ab_BGF1', 'pw_ab_BGF2', 'pw_ab_BGF3', 'pw_ab_beta',
    'pw_theta', 'pw_BGF1', 'pw_BGF2', 'pw_BGF3', 'pw_beta',
    'pw_ab_canonic_theta', 'pw_ab_canonic_alpha1', 'pw_ab_canonic_alpha2',
    'pw_ab_canonic_beta1', 'pw_ab_canonic_beta2', 'pw_ab_canonic_beta3', 'pw_ab_canonic_gamma',
    'pw_canonic_theta', 'pw_canonic_alpha1', 'pw_canonic_alpha2',
    'pw_canonic_beta1', 'pw_canonic_beta2', 'pw_canonic_beta3', 'pw_canonic_gamma',
    'osc_pw_ab_theta', 'osc_pw_ab_BGF1', 'osc_pw_ab_BGF2', 'osc_pw_ab_BGF3', 'osc_pw_ab_beta',
    'osc_pw_theta', 'osc_pw_BGF1', 'osc_pw_BGF2', 'osc_pw_BGF3', 'osc_pw_beta',
    'osc_pw_ab_canonic_theta', 'osc_pw_ab_canonic_alpha1', 'osc_pw_ab_canonic_alpha2',
    'osc_pw_ab_canonic_beta1', 'osc_pw_ab_canonic_beta2', 'osc_pw_ab_canonic_beta3', 'osc_pw_ab_canonic_gamma',
    'osc_pw_canonic_theta', 'osc_pw_canonic_alpha1', 'osc_pw_canonic_alpha2',
    'osc_pw_canonic_beta1', 'osc_pw_canonic_beta2', 'osc_pw_canonic_beta3', 'osc_pw_canonic_gamma'
]

category_boundaries = [
    'exponent', 'pw_ab_beta', 'pw_beta', 'pw_ab_canonic_gamma',
    'pw_canonic_gamma', 'osc_pw_ab_beta', 'osc_pw_beta',
    'osc_pw_ab_canonic_gamma', 'osc_pw_canonic_gamma'
]

category_labels = [
    "Parameters ",
    "Unadjusted Spectral\nAb. power ",
    "Unadjusted Spectral\nRel. power ",
    "Unadjusted Spectral\nAb. power - canonic bands",
    "Unadjusted Spectral\nRel. power - canonic bands",
    "Oscillatory spectral\nAb. power",
    "Oscillatory spectral\nRel. power",
    "Oscillatory spectral Ab. power\ncanonic bands",
    "Oscillatory spectral Rel. power\ncanonic bands"
]

# Preparar figura
fig, axes = plt.subplots(nrows=6, ncols=1, figsize=(len(features) * 0.5, 12), gridspec_kw={'height_ratios': [0.7]+ [2]*5})
#fig, axes = plt.subplots(nrows=9, ncols=1, figsize=(len(features) * 0.5, 22),
#                         gridspec_kw={'height_ratios': [0.7] + [2]*8})

# Eje para los títulos de categoría (primera fila vacía)
axes[0].axis("off")



# Dibujar heatmaps
for ax, (title, mat) in zip(axes[1:], binary_matrices.items()):
    orden_features = [f for f in features if f in mat.columns]
    mat = mat[orden_features]
    cmap = ['moccasin', 'orange']

    sns.heatmap(mat, cmap=cmap, cbar=False, ax=ax, linewidths=0.2, linecolor='whitesmoke')
    
    xticks = mat.columns.tolist()
    for boundary in category_boundaries:
        if boundary in xticks:
            idx = xticks.index(boundary) + 1
            ax.axvline(idx, color='black', linewidth=2)
    # Dibujar etiquetas de categoría SOLO en el PRIMER heatmap
   
    x_positions = []
    valid_labels = []

    for i in range(len(category_boundaries)):
        if category_boundaries[i] in xticks:
            start = 0 if i == 0 else (
                xticks.index(category_boundaries[i - 1]) + 1
                if category_boundaries[i - 1] in xticks else 0
            )
            end = xticks.index(category_boundaries[i])
            midpoint = (start + end) / 2
            x_positions.append(midpoint)
            valid_labels.append(category_labels[i])

    for xpos, label in zip(x_positions, valid_labels):
        ax.text(xpos+1, -0.7, label, ha='center', va='bottom', fontsize=10)

    ax.set_title(title, fontsize=12, pad=45, fontweight='bold', loc='left')  # Aumentá pad a tu gusto (10, 15, 20...)

    ax.set_ylabel("ROI", fontsize=14, fontweight='bold')

    # Ocultar X excepto en el último
    if ax != axes[-1]:
        ax.set_xticklabels([])
        ax.set_xlabel("")
    else:
        ax.set_xlabel("Feature", fontsize=14, fontweight='bold')

    ax.tick_params(axis='x', labelrotation=90)


axes[-1].set_xlabel("Feature", fontsize=11)
plt.subplots_adjust(hspace=1.0, top=0.95, bottom=0.15)
fig.suptitle("Performance of Harmonization: Levene test", fontsize=16, y=0.958, fontweight='bold')


import matplotlib.patches as mpatches

# Definir colores usados en los heatmaps

orange_patch = mpatches.Patch(color='orange', label='p < 0.05')
white_patch = mpatches.Patch(color='moccasin', edgecolor='black', label='p ≥ 0.05')

# Añadir leyenda global a la figura
fig.legend(handles=[ orange_patch, white_patch],
           loc='upper center', ncol=5, bbox_to_anchor=(0.5, 0.947), fontsize=12, frameon=False)
plt.savefig("D:/MulticentersEEG/graphs_thesis_LMZS/Levene_test_neuroharmonize_recombat.pdf", dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import re

# Cargar el archivo
df = pd.read_excel(r"D:\MulticentersEEG\Features_2_normativeModel\gamma_40\24_BEST_EPOCHS\AIF_Babiloni\results_harmonize\neuroharmonize\statical_tests\tests.xlsx")
df_recombat = pd.read_excel(r"D:\MulticentersEEG\Features_2_normativeModel\gamma_40\24_BEST_EPOCHS\AIF_Babiloni\results_harmonize\recombat\statical_tests\tests.xlsx")

# Limpiar nombres de features
df['ROI'] = df['Feature'].str.extract(r'roi([A-Z]+)') 
df['Feature_clean'] = df['Feature'].apply(lambda x: re.sub(r'_?roi[A-Z]+_?', '', x))

df_recombat['ROI'] = df_recombat['Feature'].str.extract(r'roi([A-Z]+)') 
df_recombat['Feature_clean'] = df_recombat['Feature'].apply(lambda x: re.sub(r'_?roi[A-Z]+_?', '', x))
# Simulación del DataFrame con columnas necesarias y algunas filas de ejemplo
np.random.seed(0)
rois = ['F', 'C', 'P', 'O','PO']
features = [
    'IAFp', 'exponent',
    'pw_ab_theta', 'pw_ab_BGF1', 'pw_ab_BGF2', 'pw_ab_BGF3', 'pw_ab_beta',
    'pw_theta', 'pw_BGF1', 'pw_BGF2', 'pw_BGF3', 'pw_beta',
    'pw_ab_canonic_theta', 'pw_ab_canonic_alpha1', 'pw_ab_canonic_alpha2',
    'pw_ab_canonic_beta1', 'pw_ab_canonic_beta2', 'pw_ab_canonic_beta3', 'pw_ab_canonic_gamma',
    'pw_canonic_theta', 'pw_canonic_alpha1', 'pw_canonic_alpha2',
    'pw_canonic_beta1', 'pw_canonic_beta2', 'pw_canonic_beta3', 'pw_canonic_gamma',
    'osc_pw_ab_theta', 'osc_pw_ab_BGF1', 'osc_pw_ab_BGF2', 'osc_pw_ab_BGF3', 'osc_pw_ab_beta',
    'osc_pw_theta', 'osc_pw_BGF1', 'osc_pw_BGF2', 'osc_pw_BGF3', 'osc_pw_beta',
    'osc_pw_ab_canonic_theta', 'osc_pw_ab_canonic_alpha1', 'osc_pw_ab_canonic_alpha2',
    'osc_pw_ab_canonic_beta1', 'osc_pw_ab_canonic_beta2', 'osc_pw_ab_canonic_beta3', 'osc_pw_ab_canonic_gamma',
    'osc_pw_canonic_theta', 'osc_pw_canonic_alpha1', 'osc_pw_canonic_alpha2',
    'osc_pw_canonic_beta1', 'osc_pw_canonic_beta2', 'osc_pw_canonic_beta3', 'osc_pw_canonic_gamma'
]

category_boundaries = [
    'exponent', 'pw_ab_beta', 'pw_beta', 'pw_ab_canonic_gamma',
    'pw_canonic_gamma', 'osc_pw_ab_beta', 'osc_pw_beta',
    'osc_pw_ab_canonic_gamma', 'osc_pw_canonic_gamma'
]

n_samples = len(rois) * len(features)

# Clasificación del cambio
def classify_change(r_harm, r_unharm, p_harm, p_unharm, epsilon=0.01):
    delta = r_harm - r_unharm
    if delta > epsilon:
        color = 1  # verde
    elif delta < -epsilon:
        color = -1  # rojo
    else:
        color = 0  # blanco
    star = f'{r_harm:.2f}*' if p_harm < 0.05 and p_unharm < 0.05 else ''

    
    return color, star

# Crear matrices por grupo
def create_matrix(df, group):
    matrix = {}
    stars = {}
    for _, row in df.iterrows():
        roi = row['ROI']
        feat = row['Feature_clean']
        r1 = row[f'rspearman_{group}_unharmonized']
        r2 = row[f'rspearman_{group}_harmonized']
        p1 = row[f'pvalue_{group}_unharmonized']
        p2 = row[f'pvalue_{group}_harmonized']
        color, star = classify_change(r2, r1, p2, p1)
        matrix.setdefault(roi, {})[feat] = color
        stars.setdefault(roi, {})[feat] = star
    return pd.DataFrame(matrix).T, pd.DataFrame(stars).T

df_age = df[df['covars'] == 'SITE_age']
df_age_recombat = df_recombat[df_recombat['covars'] == 'SITE_age_group']

hc_df, hc_annot = create_matrix(df_age, "hc")
mci_df, mci_annot = create_matrix(df_age, "mci")

hc_df_recombat, hc_annot_recombat = create_matrix(df_age_recombat, "hc")
mci_df_recombat, mci_annot_recombat = create_matrix(df_age_recombat, "mci")

df_sex = df[df['covars'] == 'SITE_age_sex']
df_sex_recombat = df_recombat[df_recombat['covars'] == 'SITE_age_sex_group']

hc_df_sex, hc_annot_sex = create_matrix(df_sex, "hc")
mci_df_sex, mci_annot_sex = create_matrix(df_sex, "mci")
hc_df_sex_recombat, hc_annot_sex_recombat = create_matrix(df_sex_recombat, "hc")
mci_df_sex_recombat, mci_annot_sex_recombat = create_matrix(df_sex_recombat, "mci")



# Reordenar columnas
#Nueroharmonize
hc_df = hc_df[features]
hc_annot = hc_annot[features]
mci_df = mci_df[features]
mci_annot = mci_annot[features]

hc_df_sex = hc_df_sex[features]
hc_annot_sex = hc_annot_sex[features]
mci_df_sex = mci_df_sex[features]
mci_annot_sex = mci_annot_sex[features]

#Recombat
hc_df_recombat = hc_df_recombat[features]
hc_annot_recombat = hc_annot_recombat[features]
mci_df_recombat = mci_df_recombat[features]
mci_annot_recombat = mci_annot_recombat[features]

hc_df_sex_recombat = hc_df_sex_recombat[features]
hc_annot_sex_recombat = hc_annot_sex_recombat[features]
mci_df_sex_recombat = mci_df_sex_recombat[features]
mci_annot_sex_recombat = mci_annot_sex_recombat[features]


# Graficar
fig, axes = plt.subplots(4, 1, figsize=(len(features)*0.5, 16), sharex=True)
data_list_1 = [hc_df, mci_df, hc_df_recombat, mci_df_recombat]
annot_list_1 = [hc_annot, mci_annot, hc_annot_recombat, mci_annot_recombat]
titles_1 = [
    "A) Change in Spearman Correlation - Neuroharmonize - HC (Age)",
    "B) Change in Spearman Correlation - Neuroharmonize - MCI (Age)",
    "C) Change in Spearman Correlation - ReComBat - HC (Age + Diagnosis)",
    "D) Change in Spearman Correlation - ReComBat - MCI (Age + Diagnosis)"
]

# tu código de visualización aquí para fig2


for ax, data, annot, title in zip(axes, data_list_1, annot_list_1, titles_1):
    cmap = ['wheat', 'white', 'darkorange'] if "Neuroharmonize" in title else ['powderblue', 'white', 'cornflowerblue']
    sns.heatmap(data, ax=ax, cmap=cmap, center=0, annot=annot, fmt='', linewidths=0.05, linecolor='whitesmoke', cbar=False)
    
    for text in ax.texts:
        text.set_rotation(90)

    ax.set_title(title, fontsize=14, pad=50, fontweight='bold', loc='left')
    for boundary in category_boundaries:
        if boundary in data.columns:
            idx = data.columns.get_loc(boundary) + 1
            ax.axvline(idx, color='black', linewidth=2)
    x_positions = []
    valid_labels = []

    for i in range(len(category_boundaries)):
        if category_boundaries[i] in xticks:
            start = 0 if i == 0 else (
                xticks.index(category_boundaries[i - 1]) + 1
                if category_boundaries[i - 1] in xticks else 0
            )
            end = xticks.index(category_boundaries[i])
            midpoint = (start + end) / 2
            x_positions.append(midpoint)
            valid_labels.append(category_labels[i])

    for xpos, label in zip(x_positions, valid_labels):
        ax.text(xpos+0.5, -0.4, label, ha='center', va='bottom', fontsize=10)

    ax.set_ylabel("ROI", fontsize=14, fontweight='bold')
    ax.tick_params(axis='x', labelrotation=90)

axes[-1].set_xlabel("Feature", fontsize=12)
import matplotlib.patches as mpatches

# Parches para la leyenda
no_change_patch = mpatches.Patch(color='white', label='No change in correlation')
increase_patch_neuro = mpatches.Patch(color='darkorange', label='r increased (NeuroHarmonize)')
decrease_patch_neuro = mpatches.Patch(color='wheat', label='r decreased (NeuroHarmonize)')
increase_patch_recombat = mpatches.Patch(color='cornflowerblue', label='r increased (ReComBat)')
decrease_patch_recombat = mpatches.Patch(color='powderblue', label='r decreased (ReComBat)')

# Mostrar la leyenda (ajusta bbox si es necesario)
fig.legend(
    handles=[
        no_change_patch, increase_patch_neuro, decrease_patch_neuro,
        increase_patch_recombat, decrease_patch_recombat
    ],
    loc='lower center',
    ncol=5,
    bbox_to_anchor=(0.5, 0.96),
    frameon=False,
    fontsize=12
)
fig.suptitle("Performance of Harmonization: Spearman Correlation", fontsize=16, y=1, fontweight='bold')


plt.tight_layout()
plt.savefig("D:/MulticentersEEG/graphs_thesis_LMZS/spearman_age.pdf", dpi=300, bbox_inches='tight')
plt.show()

# tu código de visualización aquí para fig1
fig, axes = plt.subplots(4, 1, figsize=(len(features)*0.5, 16), sharex=True)
data_list_2 = [hc_df_sex, mci_df_sex, hc_df_sex_recombat, mci_df_sex_recombat]
annot_list_2 = [hc_annot_sex, mci_annot_sex, hc_annot_sex_recombat, mci_annot_sex_recombat]
titles_2 = [
    "A) Change in Spearman Correlation - Neuroharmonize - HC (Age+Sex)",
    "B) Change in Spearman Correlation - Neuroharmonize - MCI (Age+Sex)",
    "C) Change in Spearman Correlation - ReComBat - HC (Age+Sex+Diagnosis)",
    "D) Change in Spearman Correlation - ReComBat - MCI (Age+Sex+Diagnosis)"
]

for ax, data, annot, title in zip(axes, data_list_2, annot_list_2, titles_2):
    cmap = ['wheat', 'white', 'darkorange'] if "Neuroharmonize" in title else ['powderblue', 'white', 'cornflowerblue']
    sns.heatmap(data, ax=ax, cmap=cmap, center=0, annot=annot, fmt='', linewidths=0.05, linecolor='whitesmoke', cbar=False)
    
    for text in ax.texts:
        text.set_rotation(90)

    ax.set_title(title, fontsize=14, pad=50, fontweight='bold', loc='left')
    for boundary in category_boundaries:
        if boundary in data.columns:
            idx = data.columns.get_loc(boundary) + 1
            ax.axvline(idx, color='black', linewidth=2)
    ax.set_ylabel("ROI", fontsize=14, fontweight='bold')
    x_positions = []
    valid_labels = []

    for i in range(len(category_boundaries)):
        if category_boundaries[i] in xticks:
            start = 0 if i == 0 else (
                xticks.index(category_boundaries[i - 1]) + 1
                if category_boundaries[i - 1] in xticks else 0
            )
            end = xticks.index(category_boundaries[i])
            midpoint = (start + end) / 2
            x_positions.append(midpoint)
            valid_labels.append(category_labels[i])

    for xpos, label in zip(x_positions, valid_labels):
        ax.text(xpos+0.5, -0.4, label, ha='center', va='bottom', fontsize=10)

    ax.tick_params(axis='x', labelrotation=90)

axes[-1].set_xlabel("Feature", fontsize=12)
import matplotlib.patches as mpatches

# Parches para la leyenda
no_change_patch = mpatches.Patch(color='white', label='No change in correlation')
increase_patch_neuro = mpatches.Patch(color='darkorange', label='r increased (NeuroHarmonize)')
decrease_patch_neuro = mpatches.Patch(color='wheat', label='r decreased (NeuroHarmonize)')
increase_patch_recombat = mpatches.Patch(color='cornflowerblue', label='r increased (ReComBat)')
decrease_patch_recombat = mpatches.Patch(color='powderblue', label='r decreased (ReComBat)')

# Mostrar la leyenda (ajusta bbox si es necesario)
fig.legend(
    handles=[
        no_change_patch, increase_patch_neuro, decrease_patch_neuro,
        increase_patch_recombat, decrease_patch_recombat
    ],
    loc='lower center',
    ncol=5,
    bbox_to_anchor=(0.5, 0.96),
    frameon=False,
    fontsize=12
)
fig.suptitle("Performance of Harmonization: Spearman Correlation", fontsize=16, y=1, fontweight='bold')
plt.tight_layout()
plt.savefig("D:/MulticentersEEG/graphs_thesis_LMZS/spearman_age_sex.pdf", dpi=300, bbox_inches='tight')
plt.show()
